# Docker Hands-On Exercise · Edge-Computing Notebook

The goal is to simulate a small edge-computing system:

```text
Fake Sensor Container -> Edge Processor Container -> Local Logs -> Dashboard
```

The fake sensor generates readings like:

```json
{
  "temperature": 34.2,
  "motion": true,
  "battery": 76
}
```

The edge processor receives the readings, classifies the risk locally, stores logs, and serves a small dashboard.

Example risk logic:

```text
temperature >= 35 -> HIGH
temperature >= 30 -> MEDIUM
otherwise        -> LOW
```

## How this notebook works · cells and a Jupyter terminal

Some Docker steps run once and return (building an image, checking a version). Those are ordinary **notebook cells** · just run them here.

Other steps are meant to run live and never return on their own: watching a sensor stream readings, `docker compose up` with live logs, `docker stats`, opening a shell inside a container, or stopping one service while watching another. A notebook cell cannot do those · it would hang the kernel, and it has no interactive terminal. For those steps this notebook **writes a small script into your lab folder and asks you to run it in a Jupyter terminal.**

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Terminal]** · open a Jupyter terminal and run the given command.

### Opening a Jupyter terminal

The terminal runs a real shell **on the edge device**, the same machine as the Docker daemon, so every Docker command works exactly as written.

- **JupyterLab / Notebook 7:** click the blue **+** (Launcher), then the **Terminal** tile · or **File ▸ New ▸ Terminal**.
- **Classic Notebook:** from the file browser tab, **New ▸ Terminal**.
- You can open **several terminals** at once. Part 7 needs two side by side.

If terminals are disabled on your device, you can instead SSH into the device in a separate window · every helper script this notebook writes runs the same way there.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So Part 1 writes `~/edgeDockerLab/labEnv.sh`, and every helper script starts with `source ~/edgeDockerLab/labEnv.sh`. Your dashboard port is derived automatically from your username, so both the notebook and the terminals agree with no manual step.

### Helper scripts this notebook creates

| Script | Used in | What it does |
|---|---|---|
| `labEnv.sh` | all | exports `PORT`, `USER`, `NVIDIA_IMAGE` |
| `sensorBasic/runSensor.sh` | Part 4 | build + run the sensor in the foreground (live output) |
| `sensorBasic/runSensorLogs.sh` | Part 5 | run the sensor with a persistent volume |
| `sensorBasic/inspectData.sh` | Part 5 | open a shell in the running sensor to read the log |
| `edgeComposeLab/runPipeline.sh` | Parts 6-7 | `docker compose up --build` (live) |
| `edgeComposeLab/stopPipeline.sh` | Parts 6-7 | `docker compose down` |
| `edgeComposeLab/stopProcessor.sh` | Part 7 | stop only the processor service |
| `edgeComposeLab/startProcessor.sh` | Part 7 | start the processor again |
| `watchStats.sh` | Part 8 | live `docker stats` |
| `runConstrained.sh` | Part 8 | run the sensor under CPU/memory limits |
| `cleanup.sh` | Cleanup | remove your containers, images, volumes |

**Requirements:** run this notebook with a Python 3 kernel on the edge device, where Docker is available to your user. Parts 9-11 additionally require the NVIDIA runtime and a Jetson image.

---
## Part 1 · Set Your Assigned Port

Docker resources run through the same Docker daemon on the device, so host ports are shared across the entire device. Each student or group must use a unique port.

```text
student01 -> 5101
student02 -> 5102
student03 -> 5103
student04 -> 5104
```

**[Notebook cell]** Your dashboard port is assigned automatically from the digits in your username, so `student07` gets port `5107`. This keeps everyone on a shared device from colliding, and you normally do not edit anything · just run the cell. This primer uses base **5100** rather than the 5000 used by Lab 02, so its dashboard never collides with Lab 02's. To force a specific port, set `portOverride`. The cell also writes `labEnv.sh`, which every terminal helper script reads.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook), then set up this lab's identity, port, and labEnv.sh.
import os
from labHelpers import *

# ---- OPTIONAL OVERRIDES ----------------------------------------------------
# Leave portOverride = None to auto-assign your dashboard port from the digits
# in your username (recommended on a shared/class server). Set an int to force.
portOverride = None
# Base container image. Default targets Jetson Thor (JetPack 7 / CUDA 13).
# For Jetson Orin (JetPack 6) use: "nvcr.io/nvidia/l4t-base:r36.2.0"
nvidiaImage = "nvcr.io/nvidia/cuda:13.0.0-devel-ubuntu24.04"
# ---------------------------------------------------------------------------

# This primer uses port base 5100 so it never contends with Lab 02, which uses
# 5000. student07 -> 5107.
labEnv = setupLab(
    "edgeDockerLab",
    ports={"PORT": 5100},
    portOverrides={"PORT": portOverride} if portOverride is not None else None,
    extraEnv={"NVIDIA_IMAGE": nvidiaImage},
)
userName = labEnv["USER"]
labPort = labEnv["PORT"]

print("userName     =", userName)
print("PORT         =", labPort)
print("NVIDIA_IMAGE =", nvidiaImage)


### Shared lab toolkit

The cell above imports `labHelpers.py`, which ships in the course repo next to this notebook. It provides the syntax-highlighted file cards, the env card, the rich `docker` tables, and the `preflight` / `checkpoint` / `check` machinery used throughout every lab in this course. This primer used to carry its own private copy of those helpers; it now uses the same toolkit as Labs 01-09.

**[Notebook cell]** Confirm the shell and the env file agree, then move into the lab folder:

In [ ]:
# Show the shared environment file as a readable card (replaces the raw echo/cat).
labEnvPath = os.path.expanduser("~/edgeDockerLab/labEnv.sh")
with open(labEnvPath) as fileHandle:
    labEnvText = fileHandle.read()

showEnvCard(labEnvText, title="labEnv.sh",
            envVars={"PORT": os.environ.get("PORT", "?"),
                     "USER": os.environ.get("USER", "?")})


### Preflight · check your environment

Run this once at the start. It confirms the Docker daemon and Compose are working, whether the NVIDIA runtime is available (needed for Parts 9-10), and shows the `USER` / `PORT` this notebook derived for you. Fix any failing check before continuing.

In [ ]:
import os

preflight(
    [
        check("docker daemon reachable", dockerDaemonUp(),
              hint="Docker is not reachable from your account - ask your instructor to "
                   "add you to the docker group or start the daemon.",
              link="https://docs.docker.com/engine/daemon/troubleshoot/",
              linkText="Docker daemon troubleshooting"),
        check("docker compose available", composeAvailable(),
              hint="Install the Compose plugin, or use 'docker-compose' if that is what "
                   "this device provides.",
              link="https://docs.docker.com/compose/install/",
              linkText="Install Compose"),
        check("nvidia runtime", nvidiaRuntimeAvailable(),
              hint="Parts 9-10 need the NVIDIA container runtime; Parts 1-8 work without "
                   "it. Ask your instructor if it is missing.",
              link="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html",
              linkText="NVIDIA Container Toolkit"),
        check("PORT assigned", envVarSet("PORT"),
              hint="Re-run the setupLab cell above."),
        check("lab folder created", dirExists("~/edgeDockerLab"),
              hint="Re-run the setupLab cell above; it creates ~/edgeDockerLab."),
    ],
    infoRows=[("USER", os.environ.get("USER", "?")),
              ("PORT", os.environ.get("PORT", "?")),
              ("NVIDIA_IMAGE", os.environ.get("NVIDIA_IMAGE", "?"))],
)


In [ ]:
%cd ~/edgeDockerLab

### Checkpoint · Part 1

In [ ]:
import os
userName = os.environ.get("USER", "student01")
checkpoint("Part 1 - lab environment ready", [
    check("PORT is set", envVarSet("PORT"),
          hint="Re-run the setupLab cell at the top of the notebook."),
    check("NVIDIA_IMAGE is set", envVarSet("NVIDIA_IMAGE"),
          hint="Re-run the setupLab cell; it exports NVIDIA_IMAGE."),
    check("lab folder exists", dirExists("~/edgeDockerLab"),
          hint="setupLab creates ~/edgeDockerLab. Re-run it."),
    check("labEnv.sh written", fileExists("~/edgeDockerLab/labEnv.sh"),
          hint="setupLab writes labEnv.sh so terminal scripts can source it."),
])


---
## Part 2 · Verify Docker Works on the Edge Device

**[Notebook cell]**

In [ ]:
# Verify Docker on the edge device · versions as a panel, then a hello-world check.
dockerVersions()


The original lab opens an interactive shell (`docker run -it ubuntu bash`) and runs `cat /etc/os-release`. That needs a TTY, so from a **notebook cell** we get the same result non-interactively. (If you want the real interactive experience, run `docker run --rm -it ubuntu bash` in a **[Terminal]**.)

In [ ]:
!docker run --rm ubuntu cat /etc/os-release

This shows that a container is an isolated environment running on the edge device.

### Checkpoint · Part 2

In [ ]:
checkpoint("Part 2 - Docker verified on the edge device", [
    check("docker daemon reachable", dockerDaemonUp(),
          hint="Ask your instructor to add you to the docker group.",
          link="https://docs.docker.com/engine/daemon/troubleshoot/",
          linkText="Docker daemon troubleshooting"),
    check("docker compose available", composeAvailable(),
          hint="The Compose plugin is missing on this device."),
    check("hello-world image pulled", imageExists("hello-world"),
          hint="Re-run the dockerVersions() cell - it runs a hello-world container."),
])


---
## Part 3 · Run a Local Edge Dashboard Container

NGINX is a server, so running it detached (`-d`) is correct even in production. These are fine as **notebook cells**.

In [ ]:
# Remove any earlier dashboard first so re-running this cell is safe (idempotent).
!docker rm -f ${USER}-edgeDashboard 2>/dev/null || true
!docker run -d \
  -p $PORT:80 \
  --name ${USER}-edgeDashboard \
  nginx


Open the dashboard in a browser using the device IP address and your assigned port:

```text
http://DEVICE_ADDRESS:PORT      e.g.  http://192.168.1.50:5101
```

In [ ]:
# Running containers as a table, then this dashboard's recent logs.
dockerPs()
dockerLogs(f"{os.environ.get('USER', 'student01')}-edgeDashboard", tail=20)


### Checkpoint · Part 3

Run this **while the dashboard is still up** — the next cell removes it.

In [ ]:
import os
userName = os.environ.get("USER", "student01")
dashPort = os.environ.get("PORT", "5101")
checkpoint("Part 3 - dashboard container serving", [
    check("dashboard container running", containerRunning(userName + "-edgeDashboard"),
          hint="Re-run the 'docker run -d ... nginx' cell in Part 3.",
          link="https://docs.docker.com/reference/cli/docker/container/run/",
          linkText="docker run reference"),
    check("your port is listening", portListening(dashPort),
          hint="Nothing is answering on your PORT - re-run the docker run cell and read "
               "its output for a port conflict."),
    check("NGINX answers on the port", httpOk("http://127.0.0.1:" + str(dashPort), expectText="nginx"),
          hint="The container is up but not serving. Check 'docker logs' for the container."),
])


In [ ]:
# Stop and remove the dashboard · safe to run even if it is already gone.
!docker rm -f ${USER}-edgeDashboard 2>/dev/null || true


This simulates a small dashboard hosted directly on an edge device.

---
## Part 4 · Build a Fake Sensor Container

**[Notebook cell]** Create the folder and write the source files:

In [ ]:
!mkdir -p ~/edgeDockerLab/sensorBasic
%cd ~/edgeDockerLab/sensorBasic

In [ ]:
%%writefile sensorSim.py
import time
import random
from datetime import datetime

while True:
    reading = {
        "timestamp": datetime.utcnow().isoformat(),
        "temperature": round(random.uniform(22, 40), 2),
        "motion": random.choice([True, False]),
        "battery": random.randint(20, 100),
    }

    print(reading, flush=True)
    time.sleep(2)

In [ ]:
# Preview the sensor script we just wrote.
showFile("~/edgeDockerLab/sensorBasic/sensorSim.py", language="python")

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY sensorSim.py .

CMD ["python", "sensorSim.py"]

In [ ]:
# Preview the Dockerfile we just wrote.
showFile("~/edgeDockerLab/sensorBasic/Dockerfile", language="docker")

**[Notebook cell]** Build the image:

> **Naming reminder:** the image tag stays lowercase (`${USER}-edge-sensor`) while the container it runs is camelCase (`${USER}-edgeSensor1`). Docker requires image names to be lowercase; container names may use capitals. See Lab 02, Part 13 for the full explanation.

In [ ]:
!docker build -t ${USER}-edge-sensor .

**[Notebook cell]** Write the run script:

In [ ]:
%%writefile runSensor.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh
cd ~/edgeDockerLab/sensorBasic

docker build -t "${USER}-edge-sensor" .
echo "Sensor starting in the foreground. Press Ctrl-C to stop."
docker run --rm --name "${USER}-edgeSensor1" "${USER}-edge-sensor"

In [ ]:
!chmod +x runSensor.sh

**[Terminal]** Open a Jupyter terminal and run the sensor live. You will see a new reading print every 2 seconds:

```bash
~/edgeDockerLab/sensorBasic/runSensor.sh
```

Press **Ctrl-C** to stop it (the container auto-removes because of `--rm`).

The fake sensor represents a local device generating data at the edge.

### Checkpoint · Part 4

In [ ]:
import os
userName = os.environ.get("USER", "student01")
checkpoint("Part 4 - fake sensor container built", [
    check("sensorSim.py written", fileExists("~/edgeDockerLab/sensorBasic/sensorSim.py"),
          hint="Re-run the %%writefile sensorSim.py cell in Part 4."),
    check("Dockerfile written", fileExists("~/edgeDockerLab/sensorBasic/Dockerfile"),
          hint="Re-run the %%writefile Dockerfile cell in Part 4."),
    check("sensor image built", imageExists(userName + "-edge-sensor"),
          hint="Re-run the docker build cell in Part 4. Image names must be lowercase.",
          link="https://docs.docker.com/reference/cli/docker/buildx/build/",
          linkText="docker build reference"),
    check("runSensor.sh written", fileExists("~/edgeDockerLab/sensorBasic/runSensor.sh"),
          hint="Re-run the %%writefile runSensor.sh cell."),
])


---
## Part 5 · Add Local Persistence with a Volume

Now the sensor also writes readings to a persistent volume on the device.

**[Notebook cell]** Rewrite the sensor to append to a log file:

In [ ]:
%cd ~/edgeDockerLab/sensorBasic

In [ ]:
%%writefile sensorSim.py
import time
import random
from datetime import datetime

logFile = "/data/sensorLog.txt"

while True:
    reading = {
        "timestamp": datetime.utcnow().isoformat(),
        "temperature": round(random.uniform(22, 40), 2),
        "motion": random.choice([True, False]),
        "battery": random.randint(20, 100),
    }

    line = str(reading)
    print(line, flush=True)

    with open(logFile, "a") as f:
        f.write(line + "\n")

    time.sleep(2)

**[Notebook cell]** Rebuild and create the named volume:

In [ ]:
!docker build -t ${USER}-edge-sensor .
!docker volume create ${USER}-primerEdgeData

**[Notebook cell]** Write the run script (sensor + volume, foreground) and the inspect script (shell into the running container):

In [ ]:
%%writefile runSensorLogs.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh
cd ~/edgeDockerLab/sensorBasic

docker build -t "${USER}-edge-sensor" .
docker volume create "${USER}-primerEdgeData" >/dev/null
echo "Sensor (with persistent volume) starting. Press Ctrl-C to stop."
docker run --rm \
  -v "${USER}-primerEdgeData:/data" \
  --name "${USER}-edgeSensorLogs" \
  "${USER}-edge-sensor"

In [ ]:
%%writefile inspectData.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh

echo "Opening a shell inside the running sensor container."
echo "Try:  cat /data/sensorLog.txt   then:  exit"
docker exec -it "${USER}-edgeSensorLogs" sh

In [ ]:
!chmod +x runSensorLogs.sh inspectData.sh

**[Terminal 1]** Start the sensor and leave it running:

```bash
~/edgeDockerLab/sensorBasic/runSensorLogs.sh
```

**[Terminal 2]** Open a second terminal and inspect the persisted data:

```bash
~/edgeDockerLab/sensorBasic/inspectData.sh
```

Inside that shell run `cat /data/sensorLog.txt`, then `exit`. Press **Ctrl-C** in Terminal 1 when finished.

This shows how edge systems often need local storage when internet or cloud access is unavailable. The data survives in the `${USER}-primerEdgeData` volume even after the container is removed.

### Checkpoint · Part 5

In [ ]:
import os
userName = os.environ.get("USER", "student01")
checkpoint("Part 5 - local persistence with a volume", [
    check("volume created", volumeExists(userName + "-primerEdgeData"),
          hint="Re-run the 'docker volume create' cell in Part 5.",
          link="https://docs.docker.com/engine/storage/volumes/",
          linkText="Docker volumes"),
    check("sensorSim.py now writes to /data", fileContains("~/edgeDockerLab/sensorBasic/sensorSim.py", "/data"),
          hint="Part 5 rewrites sensorSim.py so it logs into the mounted volume. "
               "Re-run that %%writefile cell."),
    check("runSensorLogs.sh written", fileExists("~/edgeDockerLab/sensorBasic/runSensorLogs.sh"),
          hint="Re-run the %%writefile runSensorLogs.sh cell."),
    check("inspectData.sh written", fileExists("~/edgeDockerLab/sensorBasic/inspectData.sh"),
          hint="Re-run the %%writefile inspectData.sh cell."),
])


---
## Part 6 · Multi-Container Edge Pipeline with Docker Compose

```text
Sensor Container -> Edge Processor Container -> Local Logs -> Dashboard
```

Target folder structure:

```text
edgeComposeLab/
  sensor/
    sensorSim.py
    Dockerfile
  processor/
    edgeProcessor.py
    Dockerfile
  compose.yaml
  runPipeline.sh
  stopPipeline.sh
```

**[Notebook cell]** Create the folders and write every file:

In [ ]:
!mkdir -p ~/edgeDockerLab/edgeComposeLab/sensor
!mkdir -p ~/edgeDockerLab/edgeComposeLab/processor
%cd ~/edgeDockerLab/edgeComposeLab

In [ ]:
%%writefile sensor/sensorSim.py
import time
import random
import requests
from datetime import datetime

processorURL = "http://edge-processor:5000/reading"

while True:
    reading = {
        "timestamp": datetime.utcnow().isoformat(),
        "temperature": round(random.uniform(22, 40), 2),
        "motion": random.choice([True, False]),
        "battery": random.randint(20, 100),
    }

    try:
        response = requests.post(processorURL, json=reading, timeout=2)
        print(f"sent: {reading} | response: {response.json()}", flush=True)
    except Exception as exc:
        print(f"failed to send reading: {exc}", flush=True)

    time.sleep(2)

In [ ]:
%%writefile sensor/Dockerfile
FROM python:3.11-slim

WORKDIR /app

RUN pip install requests

COPY sensorSim.py .

CMD ["python", "sensorSim.py"]

In [ ]:
%%writefile processor/edgeProcessor.py
from flask import Flask, request, jsonify
from datetime import datetime

app = Flask(__name__)

logFile = "/data/edgeReadings.txt"
latestReading = {}

@app.route("/reading", methods=["POST"])
def receiveReading():
    global latestReading

    data = request.json
    temperature = data.get("temperature", 0)
    motion = data.get("motion", False)
    battery = data.get("battery", 100)

    if temperature >= 35 or battery <= 25:
        risk = "HIGH"
    elif temperature >= 30 or motion:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    processed = {
        "receivedAt": datetime.utcnow().isoformat(),
        "temperature": temperature,
        "motion": motion,
        "battery": battery,
        "risk": risk,
    }

    latestReading = processed

    with open(logFile, "a") as f:
        f.write(str(processed) + "\n")

    print(processed, flush=True)
    return jsonify(processed)

@app.route("/", methods=["GET"])
def showDashboard():
    return f"""
    <html>
        <head>
            <title>Edge Dashboard</title>
            <meta http-equiv="refresh" content="2">
        </head>
        <body>
            <h1>Edge Dashboard</h1>
            <p>This dashboard is running locally inside a Docker container on the shared edge device.</p>
            <pre>{latestReading}</pre>
        </body>
    </html>
    """

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
%%writefile processor/Dockerfile
FROM python:3.11-slim

WORKDIR /app

RUN pip install flask

COPY edgeProcessor.py .

CMD ["python", "edgeProcessor.py"]

In [ ]:
%%writefile compose.yaml
services:
  edge-processor:
    build: ./processor
    ports:
      - "${PORT}:5000"
    volumes:
      - edge-processor-data:/data

  edge-sensor:
    build: ./sensor
    depends_on:
      - edge-processor

volumes:
  edge-processor-data:

In [ ]:
# Preview the compose file we just wrote.
showFile("~/edgeDockerLab/edgeComposeLab/compose.yaml", language="yaml")

In [ ]:
%%writefile runPipeline.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh
cd ~/edgeDockerLab/edgeComposeLab

echo "Building and starting the pipeline. Live logs below."
echo "Open the dashboard at  http://<DEVICE_IP>:${PORT}"
echo "Press Ctrl-C to stop, or run stopPipeline.sh from another terminal."
docker compose -p "${USER}-edge-lab" up --build

In [ ]:
%%writefile stopPipeline.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh
cd ~/edgeDockerLab/edgeComposeLab

docker compose -p "${USER}-edge-lab" down

In [ ]:
!chmod +x runPipeline.sh stopPipeline.sh

**[Terminal]** Start the whole pipeline with live logs from both services:

```bash
~/edgeDockerLab/edgeComposeLab/runPipeline.sh
```

Then open the dashboard in a browser (it auto-refreshes every 2 seconds):

```text
http://DEVICE_IP:PORT      e.g.  http://192.168.1.50:5101
```

Stop it with **Ctrl-C**, or from another **[Terminal]**:

```bash
~/edgeDockerLab/edgeComposeLab/stopPipeline.sh
```

### Why use `-p ${USER}-edge-lab`?

Docker Compose creates project-specific containers, networks, images, and volumes. Because students share the same Docker daemon, giving each project a unique name keyed on `${USER}` keeps everyone's Compose resources separate.

### Checkpoint · Part 6

In [ ]:
checkpoint("Part 6 - multi-container pipeline described", [
    check("compose.yaml written", fileExists("~/edgeDockerLab/edgeComposeLab/compose.yaml"),
          hint="Re-run the %%writefile compose.yaml cell in Part 6.",
          link="https://docs.docker.com/compose/compose-file/",
          linkText="Compose file reference"),
    check("sensor service Dockerfile", fileExists("~/edgeDockerLab/edgeComposeLab/sensor/Dockerfile"),
          hint="Re-run the %%writefile sensor/Dockerfile cell."),
    check("processor app written", fileExists("~/edgeDockerLab/edgeComposeLab/processor/edgeProcessor.py"),
          hint="Re-run the %%writefile processor/edgeProcessor.py cell."),
    check("processor exposes /reading", fileContains("~/edgeDockerLab/edgeComposeLab/processor/edgeProcessor.py", "/reading"),
          hint="edgeProcessor.py should define the POST /reading route."),
    check("runPipeline.sh written", fileExists("~/edgeDockerLab/edgeComposeLab/runPipeline.sh"),
          hint="Re-run the %%writefile runPipeline.sh cell."),
    check("stopPipeline.sh written", fileExists("~/edgeDockerLab/edgeComposeLab/stopPipeline.sh"),
          hint="Re-run the %%writefile stopPipeline.sh cell."),
])


---
## Part 7 · Simulate Edge Failure

This step genuinely needs **two terminals**, which is exactly why a Jupyter terminal is used here.

**[Notebook cell]** Write the two control scripts:

In [ ]:
%cd ~/edgeDockerLab/edgeComposeLab

In [ ]:
%%writefile stopProcessor.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh
cd ~/edgeDockerLab/edgeComposeLab

echo "Stopping only the processor service..."
docker compose -p "${USER}-edge-lab" stop edge-processor

In [ ]:
%%writefile startProcessor.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh
cd ~/edgeDockerLab/edgeComposeLab

echo "Starting the processor service again..."
docker compose -p "${USER}-edge-lab" start edge-processor

In [ ]:
!chmod +x stopProcessor.sh startProcessor.sh

**[Terminal 1]** Start the pipeline and watch the sensor logs:

```bash
~/edgeDockerLab/edgeComposeLab/runPipeline.sh
```

**[Terminal 2]** Stop only the processor, then watch Terminal 1 · the sensor begins printing `failed to send reading`:

```bash
~/edgeDockerLab/edgeComposeLab/stopProcessor.sh
```

**[Terminal 2]** Bring the processor back and watch the sensor recover:

```bash
~/edgeDockerLab/edgeComposeLab/startProcessor.sh
```

When finished, stop everything (Ctrl-C in Terminal 1, or `stopPipeline.sh`). This simulates a local edge service failure and recovery.

### Checkpoint · Part 7

In [ ]:
checkpoint("Part 7 - failure simulation scripted", [
    check("stopProcessor.sh written", fileExists("~/edgeDockerLab/edgeComposeLab/stopProcessor.sh"),
          hint="Re-run the %%writefile stopProcessor.sh cell in Part 7."),
    check("startProcessor.sh written", fileExists("~/edgeDockerLab/edgeComposeLab/startProcessor.sh"),
          hint="Re-run the %%writefile startProcessor.sh cell in Part 7."),
])


---
## Part 8 · Monitor Resource Usage on the Edge Device

`docker stats` is a live view that never returns, so it belongs in a terminal.

**[Notebook cell]** Write the monitoring scripts:

In [ ]:
%cd ~/edgeDockerLab

In [ ]:
%%writefile watchStats.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh

echo "Live resource usage · press Ctrl-C to quit."
docker stats

In [ ]:
%%writefile runConstrained.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh

echo "Running the sensor under a CPU and memory limit. Press Ctrl-C to stop."
docker run --rm \
  --memory=128m \
  --cpus=0.5 \
  --name "${USER}-edgeConstrained" \
  "${USER}-edge-sensor"

In [ ]:
!chmod +x watchStats.sh runConstrained.sh

**[Terminal]** Watch live CPU, memory, network I/O, and block I/O while a pipeline or sensor is running:

```bash
~/edgeDockerLab/watchStats.sh
```

**[Terminal]** Optional constraint test · run the sensor limited to 0.5 CPUs and 128 MB, and observe it in `watchStats.sh` from another terminal:

```bash
~/edgeDockerLab/runConstrained.sh
```

On Jetson devices you can also see system-level monitoring in a **[Terminal]** with `tegrastats` (Ctrl-C to quit). This connects Docker resource usage to the underlying edge hardware.

### Checkpoint · Part 8

In [ ]:
checkpoint("Part 8 - resource monitoring scripted", [
    check("watchStats.sh written", fileExists("~/edgeDockerLab/watchStats.sh"),
          hint="Re-run the %%writefile watchStats.sh cell in Part 8."),
    check("runConstrained.sh written", fileExists("~/edgeDockerLab/runConstrained.sh"),
          hint="Re-run the %%writefile runConstrained.sh cell in Part 8."),
    check("constrained run sets a memory cap", fileContains("~/edgeDockerLab/runConstrained.sh", "--memory"),
          hint="runConstrained.sh must pass --memory to docker run.",
          link="https://docs.docker.com/engine/containers/resource_constraints/",
          linkText="Docker resource constraints"),
])


---
## Part 9 · Build a Simple NVIDIA-Based Container

The check container runs once and exits, so these are fine as **notebook cells**. `NVIDIA_IMAGE` was set in Part 1.

> **Important:** the NVIDIA image tag must match the Jetson software stack (JetPack / L4T). If it does not match the installed version, the container may fail. Use the tag provided by the instructor.

In [ ]:
!echo "NVIDIA_IMAGE=$NVIDIA_IMAGE"
!mkdir -p ~/edgeDockerLab/nvidiaContainerCheck
%cd ~/edgeDockerLab/nvidiaContainerCheck

In [ ]:
%%writefile gpuCheck.sh
#!/bin/bash

echo "Running inside an NVIDIA-based container"
echo "Container OS:"
cat /etc/os-release

echo ""
echo "Checking NVIDIA device files:"
ls /dev/nvhost* /dev/nvidia* 2>/dev/null | head || echo "No NVIDIA device files found"

echo ""
echo "Checking CUDA folder:"
ls /usr/local/cuda 2>/dev/null || echo "CUDA folder not found in this image"

echo ""
echo "GPU check complete"

In [ ]:
%%writefile Dockerfile
ARG NVIDIA_IMAGE
FROM ${NVIDIA_IMAGE}

WORKDIR /app

COPY gpuCheck.sh .

RUN chmod +x gpuCheck.sh

CMD ["/app/gpuCheck.sh"]

**[Notebook cell]** Build the image, passing the base image as a build argument:

In [ ]:
!docker build \
  --build-arg NVIDIA_IMAGE=$NVIDIA_IMAGE \
  -t ${USER}-primer-nvidia-check .

**[Notebook cell]** Run without the NVIDIA runtime:

In [ ]:
!docker run --rm ${USER}-primer-nvidia-check

**[Notebook cell]** Run with the NVIDIA runtime and compare:

In [ ]:
!docker run --rm --runtime nvidia ${USER}-primer-nvidia-check

Using an NVIDIA base image and using the NVIDIA runtime are related but not the same thing:

- the **base image** controls the software environment inside the container
- the **NVIDIA runtime** controls access to NVIDIA hardware and runtime support from the host

With the runtime enabled you should see additional NVIDIA device files and/or the CUDA folder appear.

### Checkpoint · Part 9

In [ ]:
import os
userName = os.environ.get("USER", "student01")
checkpoint("Part 9 - NVIDIA container built", [
    check("gpuCheck.sh written", fileExists("~/edgeDockerLab/nvidiaContainerCheck/gpuCheck.sh"),
          hint="Re-run the %%writefile gpuCheck.sh cell in Part 9."),
    check("Dockerfile written", fileExists("~/edgeDockerLab/nvidiaContainerCheck/Dockerfile"),
          hint="Re-run the %%writefile Dockerfile cell in Part 9."),
    check("nvidia-check image built", imageExists(userName + "-primer-nvidia-check"),
          hint="Re-run the docker build cell in Part 9. The image tag must be lowercase - "
               "that is why it is primer-nvidia-check and not primerNvidiaCheck."),
])


---
## Part 10 · Use Docker Compose with an NVIDIA Container

The check container exits on its own, so a foreground `up` returns · fine as **notebook cells**.

In [ ]:
%cd ~/edgeDockerLab/nvidiaContainerCheck

In [ ]:
%%writefile compose.yaml
services:
  nvidia-check:
    build:
      context: .
      args:
        NVIDIA_IMAGE: ${NVIDIA_IMAGE}
    runtime: nvidia

In [ ]:
!docker compose -p ${USER}-primer-nvidia-check up --build

In [ ]:
!docker compose -p ${USER}-primer-nvidia-check down

This shows how NVIDIA-backed edge services can be described in a Compose file instead of only using long `docker run` commands.

### Checkpoint · Part 10

In [ ]:
checkpoint("Part 10 - NVIDIA service described in Compose", [
    check("compose.yaml written", fileExists("~/edgeDockerLab/nvidiaContainerCheck/compose.yaml"),
          hint="Re-run the %%writefile compose.yaml cell in Part 10."),
    check("compose requests the nvidia runtime", fileContains("~/edgeDockerLab/nvidiaContainerCheck/compose.yaml", "runtime: nvidia"),
          hint="The service needs 'runtime: nvidia' to reach the GPU.",
          link="https://docs.docker.com/compose/compose-file/05-services/",
          linkText="Compose services reference"),
])


---
## Part 11 · Connect NVIDIA Containers to Edge AI

NVIDIA containers are useful because they package GPU-related dependencies with the application environment. For edge AI this can help with:

- model inference
- CUDA dependencies
- computer vision libraries
- reproducible deployments
- moving workloads between Jetson devices, GPU servers, and cloud GPU machines

Example edge AI pipeline:

```text
USB Camera
    |
YOLO / AI Inference Container using NVIDIA GPU
    |
Edge Processor Container
    |
Local Dashboard and Logs
```

A real project could use this setup for distracted driver detection, object detection, traffic monitoring, robotics perception, industrial sensor monitoring, or smart city camera analytics.

The main idea is that the model runs **locally** on the edge device. This reduces cloud dependency, lowers latency, and can improve privacy because raw sensor or camera data does not need to leave the device.

In [ ]:
labSummary("Docker primer - progress")


---
## Cleanup

At the end of the lab, remove your containers, images, and volumes. Only remove your own resources.

**[Notebook cell]** Write a single cleanup script (you can run it here or in a **[Terminal]**):

In [ ]:
%cd ~/edgeDockerLab

In [ ]:
%%writefile cleanup.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/edgeDockerLab/labEnv.sh

# Compose resources
( cd ~/edgeDockerLab/edgeComposeLab 2>/dev/null && docker compose -p "${USER}-edge-lab" down -v ) || true
( cd ~/edgeDockerLab/nvidiaContainerCheck 2>/dev/null && docker compose -p "${USER}-primer-nvidia-check" down ) || true

# Standalone containers
docker rm -f "${USER}-edgeDashboard" 2>/dev/null || true
docker rm -f "${USER}-edgeSensor1" 2>/dev/null || true
docker rm -f "${USER}-edgeSensorLogs" 2>/dev/null || true
docker rm -f "${USER}-edgeConstrained" 2>/dev/null || true

# Images and the data volume
docker rmi "${USER}-edge-sensor" 2>/dev/null || true
docker rmi "${USER}-primer-nvidia-check" 2>/dev/null || true
docker volume rm "${USER}-primerEdgeData" 2>/dev/null || true

echo "Cleanup complete."

In [ ]:
!chmod +x cleanup.sh
!./cleanup.sh

**[Notebook cell]** Optional · remove the whole lab folder (uncomment to run):

In [ ]:
# !rm -rf ~/edgeDockerLab

Only remove your own files, containers, images, and volumes.